# COLCAP Volatility vs. Exchange Rate (TRM): Causal Inference Analysis

Preliminary statistical analysis for my undergraduate thesis, testing whether
the Colombian peso exchange rate (TRM) has a statistically significant
relationship with the COLCAP stock index, using hypothesis testing and
causal inference methods (Granger causality, cointegration) ahead of full
ML model development (LSTM, Random Forest, XGBoost) under CRISP-DM.

Data: daily COLCAP closing prices and TRM values, 2008-2026, merged on
overlapping trading days (4,524 paired observations).

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, coint
import warnings
warnings.filterwarnings('ignore')

# Load COLCAP historical data
colcap = pd.read_csv('Datos_historicos_del_COLCAP.csv', encoding='utf-8-sig')
colcap.columns = [c.strip() for c in colcap.columns]
colcap['Fecha'] = pd.to_datetime(colcap['Fecha'], format='%d.%m.%Y')

def to_num(x):
    if pd.isna(x):
        return np.nan
    return float(str(x).replace('.', '').replace(',', '.'))

colcap['Close'] = colcap['Último'].apply(to_num)
colcap = colcap[['Fecha', 'Close']].sort_values('Fecha').reset_index(drop=True)

# Load TRM exchange rate data
trm = pd.read_csv('Tasa_de_cambio_del_peso_colombiano.csv', encoding='utf-8-sig')
trm.columns = [c.strip() for c in trm.columns]
trm.rename(columns={trm.columns[0]: 'Fecha', trm.columns[1]: 'TRM'}, inplace=True)
trm['Fecha'] = pd.to_datetime(trm['Fecha'], format='%Y/%m/%d')
trm = trm.sort_values('Fecha').reset_index(drop=True)

print("COLCAP range:", colcap['Fecha'].min(), "to", colcap['Fecha'].max(), "rows:", len(colcap))
print("TRM range:", trm['Fecha'].min(), "to", trm['Fecha'].max(), "rows:", len(trm))

COLCAP range: 2008-01-11 00:00:00 to 2026-08-05 00:00:00 rows: 4525
TRM range: 1991-11-27 00:00:00 to 2026-08-06 00:00:00 rows: 12672


In [2]:
# Merge on overlapping dates and compute log returns
df = pd.merge(colcap, trm, on='Fecha', how='inner').sort_values('Fecha').reset_index(drop=True)
df['ret_colcap'] = np.log(df['Close']).diff()
df['ret_trm'] = np.log(df['TRM']).diff()
df = df.dropna().reset_index(drop=True)

print("Merged overlap range:", df['Fecha'].min(), "to", df['Fecha'].max())
print("Usable paired observations:", len(df))
df.head()

Merged overlap range: 2008-01-14 00:00:00 to 2026-08-05 00:00:00
Usable paired observations: 4524


,Fecha,Close,TRM,ret_colcap,ret_trm
0,2008-01-14,1000.00,1985.35,0.000000,-0.009220
1,2008-01-15,980.21,1949.43,-0.019988,-0.018258
2,2008-01-16,958.99,1948.91,-0.021886,-0.000267
3,2008-01-17,928.38,1960.49,-0.032440,0.005924
4,2008-01-18,903.49,1947.60,-0.027176,-0.006597


In [3]:
# 1. Stationarity (ADF test)
print("Stationarity on returns (expect stationary):")
for col in ['ret_colcap', 'ret_trm']:
    result = adfuller(df[col], autolag='AIC')
    print(f"  {col}: ADF stat={result[0]:.4f}, p-value={result[1]:.6f}, stationary={result[1]<0.05}")

print("\nStationarity on levels (expect non-stationary):")
for col in ['Close', 'TRM']:
    result = adfuller(df[col], autolag='AIC')
    print(f"  {col}: ADF stat={result[0]:.4f}, p-value={result[1]:.6f}, stationary={result[1]<0.05}")

Stationarity on returns (expect stationary):
  ret_colcap: ADF stat=-28.3230, p-value=0.000000, stationary=True


  ret_trm: ADF stat=-38.5884, p-value=0.000000, stationary=True

Stationarity on levels (expect non-stationary):
  Close: ADF stat=-1.4737, p-value=0.546462, stationary=False


  TRM: ADF stat=-1.3060, p-value=0.626413, stationary=False


In [4]:
# 2. Correlation between returns
from scipy import stats
r, p = stats.pearsonr(df['ret_colcap'], df['ret_trm'])
print(f"Pearson r={r:.4f}, p-value={p:.6f}, significant={p<0.05}")
print(f"Spearman correlation: {df['ret_colcap'].corr(df['ret_trm'], method='spearman'):.4f}")

Pearson r=-0.0303, p-value=0.041347, significant=True
Spearman correlation: -0.0493


In [5]:
# 3. Rolling correlation - has the relationship changed over time?
df_sorted = df.sort_values('Fecha')
roll = df_sorted.set_index('Fecha')[['ret_colcap', 'ret_trm']].rolling(252).corr().unstack()['ret_colcap']['ret_trm']
roll = roll.dropna()
print(f"Rolling 1yr correlation: min={roll.min():.3f}, max={roll.max():.3f}, mean={roll.mean():.3f}, latest={roll.iloc[-1]:.3f}")

Rolling 1yr correlation: min=-0.248, max=0.120, mean=-0.051, latest=-0.035


In [6]:
# 4. Granger causality (both directions, up to 5 lags)
print("Does TRM Granger-cause COLCAP?")
gc_data = df[['ret_colcap', 'ret_trm']].dropna()
gc_result = grangercausalitytests(gc_data, maxlag=5)
for lag in range(1, 6):
    pval = gc_result[lag][0]['ssr_ftest'][1]
    print(f"  Lag {lag}: p-value={pval:.6f}, significant={pval<0.05}")

print("\nDoes COLCAP Granger-cause TRM?")
gc_data_rev = df[['ret_trm', 'ret_colcap']].dropna()
gc_result_rev = grangercausalitytests(gc_data_rev, maxlag=5)
for lag in range(1, 6):
    pval = gc_result_rev[lag][0]['ssr_ftest'][1]
    print(f"  Lag {lag}: p-value={pval:.6f}, significant={pval<0.05}")

Does TRM Granger-cause COLCAP?
  Lag 1: p-value=0.000000, significant=True
  Lag 2: p-value=0.000122, significant=True
  Lag 3: p-value=0.000231, significant=True
  Lag 4: p-value=0.000034, significant=True
  Lag 5: p-value=0.000095, significant=True

Does COLCAP Granger-cause TRM?
  Lag 1: p-value=0.000000, significant=True
  Lag 2: p-value=0.000000, significant=True
  Lag 3: p-value=0.000000, significant=True
  Lag 4: p-value=0.000000, significant=True
  Lag 5: p-value=0.000000, significant=True


In [7]:
# 5. Cointegration (Engle-Granger) on price levels
score, pvalue, _ = coint(df['Close'], df['TRM'])
print(f"Engle-Granger test statistic: {score:.4f}, p-value={pvalue:.6f}, cointegrated={pvalue<0.05}")

Engle-Granger test statistic: -1.4348, p-value=0.785340, cointegrated=False


## Key findings

- Both return series are stationary (ADF p < 0.001); price levels are not, as expected for financial series.
- Correlation between COLCAP and TRM returns is statistically significant but economically weak (Pearson r = -0.03, p = 0.04).
- Granger causality is significant and bidirectional up to 5 lags in both directions.
- No long-run cointegration between price levels (Engle-Granger p = 0.79).

**Interpretation:** TRM and COLCAP show short-term shock transmission in both
directions, but no stable long-run equilibrium relationship. This is
consistent with both series responding to shared global risk sentiment
shocks rather than one directly driving the other. Next step: extend this
with LSTM, Random Forest, and XGBoost to test for non-linear predictive
relationships beyond what linear Granger causality can capture.